# KLUE-RoBERTa 파인튜닝 — 위험도 3-class / 카테고리 7-class

**실행 환경**: Google Colab (런타임 → 런타임 유형 변경 → **T4 GPU** 선택)

- 데이터: GitHub `dataset_v1` (train 643 / val 34) — raw URL로 직접 로드
- 평가 1순위 지표: **danger Recall** (스펙 §5.3 — 위험 조항을 놓치는 FN이 최악의 오류), 2순위: Macro-F1
- **test.csv는 이 노트북에서 로드하지 않는다.** 최종 비교는 3주차 말 별도 1회 실행.

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 — 런타임 유형을 GPU로 바꾸세요')

In [ ]:
import pandas as pd

BASE = 'https://raw.githubusercontent.com/aiswody/contract-guardian/main/ml/data/processed/dataset_v1'
train_df = pd.read_csv(f'{BASE}/train.csv')
val_df = pd.read_csv(f'{BASE}/val.csv')
print(len(train_df), len(val_df))
print(train_df.risk_level.value_counts())

In [ ]:
# ===== 공통 설정 =====
import numpy as np
from datasets import Dataset
from sklearn.metrics import f1_score, recall_score, classification_report
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          Trainer, TrainingArguments)

MODEL_NAME = 'klue/roberta-base'
SEED = 42
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_dataset(df, label_col, label2id):
    ds = Dataset.from_pandas(df[['text', label_col]].rename(columns={label_col: 'label_str'}))
    ds = ds.map(lambda x: {'labels': label2id[x['label_str']]})
    ds = ds.map(lambda x: tokenizer(x['text'], truncation=True, max_length=128), batched=False)
    return ds

def train_classifier(label_col, labels, output_dir, epochs=5):
    label2id = {l: i for i, l in enumerate(labels)}
    id2label = {i: l for l, i in label2id.items()}
    train_ds = make_dataset(train_df, label_col, label2id)
    val_ds = make_dataset(val_df, label_col, label2id)

    def compute_metrics(eval_pred):
        logits, y = eval_pred
        pred = logits.argmax(-1)
        m = {'macro_f1': f1_score(y, pred, average='macro', zero_division=0)}
        if 'danger' in label2id:
            d = label2id['danger']
            m['danger_recall'] = recall_score(y == d, pred == d, zero_division=0)
        return m

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(labels), label2id=label2id, id2label=id2label)
    args = TrainingArguments(
        output_dir=output_dir, seed=SEED,
        num_train_epochs=epochs, learning_rate=2e-5,
        per_device_train_batch_size=16, per_device_eval_batch_size=64,
        warmup_ratio=0.1, weight_decay=0.01,
        eval_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True, metric_for_best_model='macro_f1',
        logging_steps=20, report_to='none')
    trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                      eval_dataset=val_ds, compute_metrics=compute_metrics,
                      processing_class=tokenizer)
    trainer.train()

    pred = trainer.predict(val_ds)
    y_pred = pred.predictions.argmax(-1)
    print(classification_report(pred.label_ids, y_pred, target_names=labels, digits=3, zero_division=0))
    trainer.save_model(f'{output_dir}/best')
    tokenizer.save_pretrained(f'{output_dir}/best')
    return trainer

## 모델 A — 위험도 3-class

In [ ]:
risk_trainer = train_classifier('risk_level', ['safe', 'caution', 'danger'], 'risk_model')

## 모델 A' — 카테고리 7-class (동일 백본)

In [ ]:
CATEGORIES = ['deposit_return', 'repair_defect', 'restoration',
              'termination_renewal', 'lien_rights', 'fees_utilities', 'etc']
cat_trainer = train_classifier('category', CATEGORIES, 'category_model')

## 아티팩트 다운로드

학습이 끝나면 아래 셀로 압축해 다운로드한다. (모델 파일은 용량 문제로 git에 넣지 않는다 — CLAUDE.md 구조 참조. 로컬 `model-server/models/`에 풀어둘 것)

In [ ]:
!zip -qr models.zip risk_model/best category_model/best
from google.colab import files
files.download('models.zip')

## (봉인) test set 최종 평가 — 3주차 말 1회만 실행

아래 셀은 모든 학습·튜닝(threshold 포함)이 끝난 뒤 **단 한 번만** 실행한다.
실행 후 수치를 baseline과 함께 리포트에 기록하고, 이후 모델을 수정했다면 test 수치는 무효.

In [ ]:
# test_df = pd.read_csv(f'{BASE}/test.csv')
# ... (봉인 해제 시 작성)